# Lab 5 — RAG Continued: Indexes, Reranking, Query Transformation & Routing

**Module 2 · Lecture 5 companion lab**

---

## The one problem we will solve

> **Build "AskCorp" — an internal helpdesk assistant for a fictional company.**
>
> Employees type short, messy questions into Slack. The assistant must find the right
> passage from the company knowledge base and answer it with a citation.
>
> The knowledge base has three very different collections:
> **HR policies**, **Engineering docs**, and **Ops / infrastructure runbooks**.

Every technique from the lecture is a step towards making AskCorp work:

| Lecture topic | What it fixes in AskCorp | Section |
|---|---|---|
| Embeddings + exact search (`IndexFlatIP`) | Get a baseline that is always correct | 3–4 |
| IVF (`nlist`, `nprobe`) | 20 M vectors is too slow to scan | 5 |
| PQ / IVF+PQ | 20 M vectors is too big for RAM | 6 |
| HNSW | The graph-based alternative | 7 |
| ChromaDB + metadata filtering | "numbers but no documents"; scoped questions | 8 |
| Cross-encoder reranking | Top-5 is nearly right, but ordered badly | 9 |
| Query transformation | "OOMKilled" and "Does it apply to contractors?" | 10 |
| Query routing | Don't search HR docs for a Kubernetes bug | 11 |
| End-to-end pipeline | Put it all together | 12 |

**Runtime:** CPU is fine. No API key is required — the LLM steps have an offline stub
so the whole notebook runs top-to-bottom in a free Colab session.

---
## 1. Setup

`faiss-cpu` gives us the indexes, `sentence-transformers` gives us the bi-encoder and
cross-encoder, `chromadb` gives us the database wrapper.

In [ ]:
!pip -q install faiss-cpu sentence-transformers chromadb

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


In [ ]:
import os, re, time, json, textwrap
import numpy as np
import faiss

np.random.seed(0)

def show(title, rows):
    """Tiny pretty-printer so every result in this lab looks the same."""
    print("=" * 78)
    print(title)
    print("=" * 78)
    for r in rows:
        print(r)
    print()

def show_hits(title, hits):
    """hits = list of dicts with keys: score, id, dept, source, text"""
    print("=" * 78)
    print(title)
    print("=" * 78)
    for rank, h in enumerate(hits, 1):
        head = f"{rank}. [{h['score']:+.3f}] ({h['dept']:<4} | {h['source']})"
        body = textwrap.fill(h["text"], width=74, initial_indent="     ",
                             subsequent_indent="     ")
        print(head)
        print(body)
    print()

print("faiss version:", faiss.__version__)

faiss version: 1.15.1


---
## 2. The dataset

Deliberately tiny (36 passages) so you can *read the whole corpus* and check every
result by eye. This is the single dataset used for the entire lab.

Each record stores the three things a vector DB needs:

* **text** — the chunk the LLM will read
* **vector** — computed in the next section
* **metadata** — `dept`, `source`, `year` — what makes filtering and citation possible

In [ ]:
CORPUS = [
    # ---------------- HR ----------------
    dict(id="hr-01", dept="HR", source="hr_policy.pdf", year=2024,
         text="Full-time employees accrue 24 days of paid annual leave per calendar year. Unused leave up to 10 days may be carried into the next year."),
    dict(id="hr-02", dept="HR", source="hr_policy.pdf", year=2024,
         text="Employees are entitled to 12 paid sick days per year. A medical certificate is required for any absence longer than three consecutive days."),
    dict(id="hr-03", dept="HR", source="hr_policy.pdf", year=2024,
         text="The standard parental leave policy grants 26 weeks of paid leave to the primary caregiver and 4 weeks to the secondary caregiver."),
    dict(id="hr-04", dept="HR", source="hr_policy.pdf", year=2024,
         text="Contract and temporary staff are not covered by the standard parental leave policy. Their entitlements are defined individually in the vendor agreement."),
    dict(id="hr-05", dept="HR", source="benefits_handbook.pdf", year=2024,
         text="The employee car-leasing benefit allows staff at grade M3 and above to lease a vehicle through payroll deduction over a 36-month term."),
    dict(id="hr-06", dept="HR", source="benefits_handbook.pdf", year=2024,
         text="Health insurance covers the employee, spouse and two dependent children up to a sum insured of eight lakh rupees per year."),
    dict(id="hr-07", dept="HR", source="benefits_handbook.pdf", year=2023,
         text="In the 2023 handbook the health insurance sum insured was five lakh rupees. This figure was revised upward in the 2024 revision."),
    dict(id="hr-08", dept="HR", source="payroll_faq.md", year=2024,
         text="Salary is credited on the last working day of each month. Payroll discrepancies must be raised with the payroll desk within 30 days of the pay date."),
    dict(id="hr-09", dept="HR", source="payroll_faq.md", year=2024,
         text="Reimbursement claims for travel and internet are processed in the payroll cycle following approval by the reporting manager."),
    dict(id="hr-10", dept="HR", source="hr_policy.pdf", year=2024,
         text="The hybrid work policy requires employees to be present in the office for a minimum of ten days per month, coordinated with their manager."),
    dict(id="hr-11", dept="HR", source="hr_policy.pdf", year=2024,
         text="Employees may request a sabbatical of up to six months after five years of continuous service, subject to business approval."),
    dict(id="hr-12", dept="HR", source="onboarding.md", year=2024,
         text="New joiners complete mandatory security and code-of-conduct training within the first two weeks of their start date."),

    # ---------------- ENGINEERING ----------------
    dict(id="en-01", dept="ENG", source="auth_service.md", year=2024,
         text="The auth module issues JWT access tokens with a 15 minute expiry and refresh tokens valid for 24 hours. Token lifetime is set by AUTH_TOKEN_TTL."),
    dict(id="en-02", dept="ENG", source="auth_service.md", year=2024,
         text="If clients report tokens expiring early, check for clock skew between the auth pods and the API gateway; skew above 60 seconds invalidates tokens prematurely."),
    dict(id="en-03", dept="ENG", source="k8s_runbook.md", year=2024,
         text="A container exits with OOMKilled when it exceeds its memory limit. Kubernetes terminates the process with exit code 137 and restarts the pod."),
    dict(id="en-04", dept="ENG", source="k8s_runbook.md", year=2024,
         text="To remediate OOMKilled pods, raise resources.limits.memory in the deployment manifest, or profile the service for memory leaks before scaling."),
    dict(id="en-05", dept="ENG", source="contributing.md", year=2024,
         text="Open a pull request against the develop branch. Every PR needs one approving review, a green CI run and a linked ticket before it can be merged."),
    dict(id="en-06", dept="ENG", source="contributing.md", year=2024,
         text="Commit messages follow the conventional commits format. Squash merging is enabled so the PR title becomes the commit message on develop."),
    dict(id="en-07", dept="ENG", source="api_guide.md", year=2024,
         text="The public API is rate limited to 1000 requests per minute per API key. Exceeding the limit returns HTTP 429 with a Retry-After header."),
    dict(id="en-08", dept="ENG", source="api_guide.md", year=2024,
         text="API error code E-4021 indicates a malformed pagination cursor. Reissue the request without the cursor to restart from the first page."),
    dict(id="en-09", dept="ENG", source="db_schema.md", year=2024,
         text="The orders table is partitioned by month. Queries that do not filter on order_date will scan every partition and are the usual cause of slow reports."),
    dict(id="en-10", dept="ENG", source="db_schema.md", year=2024,
         text="Schema migrations run through Alembic in the release pipeline. Backward incompatible column drops must be split across two releases."),
    dict(id="en-11", dept="ENG", source="logging_guide.md", year=2024,
         text="All services emit structured JSON logs with a trace_id field. Use the trace_id to follow one request across the gateway, auth and payment services."),
    dict(id="en-12", dept="ENG", source="python_style.md", year=2024,
         text="Python services target version 3.11, are formatted with black and type-checked with mypy in strict mode on the CI pipeline."),

    # ---------------- OPS ----------------
    dict(id="op-01", dept="OPS", source="incident_playbook.md", year=2024,
         text="Severity 1 incidents require an incident commander within five minutes and a status page update within fifteen minutes of declaration."),
    dict(id="op-02", dept="OPS", source="incident_playbook.md", year=2024,
         text="When the payment container crashes, first check the last deployment, then the pod events, then the upstream provider status page."),
    dict(id="op-03", dept="OPS", source="ci_pipeline.md", year=2024,
         text="A pipeline failure at the build stage is usually a dependency resolution error. Clear the runner cache and rerun before investigating further."),
    dict(id="op-04", dept="OPS", source="ci_pipeline.md", year=2024,
         text="Deployment pipelines block if the staging smoke test suite fails. The smoke tests run against the staging gateway for ninety seconds after rollout."),
    dict(id="op-05", dept="OPS", source="monitoring.md", year=2024,
         text="Latency dashboards are published per region. The Mumbai and US-East deployments are monitored on separate boards with independent alert thresholds."),
    dict(id="op-06", dept="OPS", source="monitoring.md", year=2024,
         text="The Mumbai region reported a p99 latency spike of 2.4 seconds on 14 March, traced to an undersized connection pool on the read replica."),
    dict(id="op-07", dept="OPS", source="monitoring.md", year=2024,
         text="The US-East region remained within its 400 millisecond p99 budget through March and raised no latency alerts during that window."),
    dict(id="op-08", dept="OPS", source="oncall.md", year=2024,
         text="On-call rotations run Monday to Monday. The secondary on-call engineer is paged only if the primary does not acknowledge within ten minutes."),
    dict(id="op-09", dept="OPS", source="oncall.md", year=2024,
         text="Every production incident requires a blameless postmortem published within five working days of resolution."),
    dict(id="op-10", dept="OPS", source="network_manual.pdf", year=2024,
         text="Switch association data is collected only for reporting access points. Rogue units are detected by signal only, so no switch association data exists for them."),
    dict(id="op-11", dept="OPS", source="network_manual.pdf", year=2024,
         text="Network error code NET-503 means the access point lost its uplink to the distribution switch and fell back to mesh mode."),
    dict(id="op-12", dept="OPS", source="backup_policy.md", year=2024,
         text="Production databases are snapshotted every six hours and retained for 30 days. Restore drills are performed quarterly."),
]

DOCS  = [d["text"] for d in CORPUS]
BY_ID = {d["id"]: d for d in CORPUS}

show("Corpus summary", [
    f"passages : {len(CORPUS)}",
    f"depts    : {sorted(set(d['dept'] for d in CORPUS))}",
    f"sources  : {len(set(d['source'] for d in CORPUS))}",
    f"example  : {CORPUS[0]['id']} -> {CORPUS[0]['text'][:60]}...",
])

Corpus summary
passages : 36
depts    : ['ENG', 'HR', 'OPS']
sources  : 17
example  : hr-01 -> Full-time employees accrue 24 days of paid annual leave per ...



And the questions we will keep coming back to. Notice how *badly written* most of them
are — that is exactly the production problem the lecture's second half is about.

In [ ]:
QUESTIONS = [
    "How many days of leave do I have left?",   # easy, HR
    "OOMKilled",                                 # jargon fragment, ENG
    "Does it apply to contractors?",             # needs conversation history
    "Pipeline failed.",                          # too vague for keywords
    "Did the latency spike happen in both the Mumbai and US-East deployments?",
    "Why can't I pull switch association data for the rogue units on Floor 3?",
]
for q in QUESTIONS:
    print(" *", q)

 * How many days of leave do I have left?
 * OOMKilled
 * Does it apply to contractors?
 * Pipeline failed.
 * Did the latency spike happen in both the Mumbai and US-East deployments?
 * Why can't I pull switch association data for the rogue units on Floor 3?


---
## 3. Embeddings — the bi-encoder

The **bi-encoder** encodes the query and the document **separately**, so every document
vector can be computed once, offline. Search then reduces to a similarity comparison.
That separation is the whole reason retrieval scales.

We normalise the vectors to unit length. After normalisation, **inner product = cosine
similarity**, which is why we will use `IndexFlatIP` rather than `IndexFlatL2`.

In [ ]:
from sentence_transformers import SentenceTransformer

bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")

def embed(texts, normalize=True):
    """Encode a list of strings into a float32 matrix of shape (n, d)."""
    v = bi_encoder.encode(list(texts), convert_to_numpy=True,
                          normalize_embeddings=normalize,
                          show_progress_bar=False)
    return v.astype("float32")

DOC_VECS = embed(DOCS)
DIM = DOC_VECS.shape[1]

show("Embedding matrix", [
    f"shape          : {DOC_VECS.shape}   (n_docs, dim)",
    f"dtype          : {DOC_VECS.dtype}",
    f"norm of row 0  : {np.linalg.norm(DOC_VECS[0]):.4f}   (1.0 => cosine == dot product)",
    f"raw size       : {DOC_VECS.nbytes / 1024:.1f} KB for {len(CORPUS)} passages",
])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding matrix
shape          : (36, 384)   (n_docs, dim)
dtype          : float32
norm of row 0  : 1.0000   (1.0 => cosine == dot product)
raw size       : 54.0 KB for 36 passages



### 3.1 Similarity without any index

Before any database: cosine similarity is just a dot product. Everything that follows is
an *optimisation* of this one line.

In [ ]:
def cosine_scores(query, doc_vecs=DOC_VECS):
    """Brute-force similarity of one query against every document."""
    q = embed([query])          # (1, d)
    return (doc_vecs @ q[0])    # (n,)

def brute_force_search(query, k=3):
    s = cosine_scores(query)
    order = np.argsort(-s)[:k]
    return [dict(score=float(s[i]), **CORPUS[i]) for i in order]

show_hits("Brute force: 'How many days of leave do I have left?'",
          brute_force_search("How many days of leave do I have left?"))

Brute force: 'How many days of leave do I have left?'
1. [+0.714] (HR   | hr_policy.pdf)
     Full-time employees accrue 24 days of paid annual leave per calendar
     year. Unused leave up to 10 days may be carried into the next year.
2. [+0.467] (HR   | hr_policy.pdf)
     Employees are entitled to 12 paid sick days per year. A medical
     certificate is required for any absence longer than three consecutive
     days.
3. [+0.450] (HR   | hr_policy.pdf)
     Employees may request a sabbatical of up to six months after five
     years of continuous service, subject to business approval.



> **Check your understanding.** The winning passage never contains the phrase
> *"days of leave"* in the way the user typed it. Keyword search (BM25) would have
> struggled; semantic search did not. That is the value the embedding adds.

---
## 4. FAISS — exact search with `IndexFlatIP`

FAISS is **a library, not a database**. It returns *ids and distances*; keeping
the id → text map is our job. Forget that map and you get the classic
*"FAISS gave me numbers but I lost my documents"* bug — so we build the map deliberately.

`IndexFlatIP` compares the query against **every** vector. No training, no tuning,
100% recall. Per the lecture's decision tree, this is the right choice up to ~1M vectors.

In [ ]:
def build_flat_ip(vecs):
    """Exact, brute-force index over unit-normalised vectors (inner product = cosine)."""
    index = faiss.IndexFlatIP(vecs.shape[1])
    index.add(vecs)                 # no train() needed for a flat index
    return index

flat_index = build_flat_ip(DOC_VECS)

# The id -> document map that FAISS does NOT keep for us.
ID_MAP = {i: CORPUS[i] for i in range(len(CORPUS))}

show("IndexFlatIP", [
    f"is_trained : {flat_index.is_trained}",
    f"ntotal     : {flat_index.ntotal}",
    f"serialized : {len(faiss.serialize_index(flat_index)) / 1024:.1f} KB",
])

IndexFlatIP
is_trained : True
ntotal     : 36
serialized : 54.0 KB



In [ ]:
def faiss_search(index, query, k=3, id_map=ID_MAP):
    """Search any FAISS index and rehydrate ids back into documents."""
    q = embed([query])
    scores, ids = index.search(q, k)          # both shape (1, k)
    hits = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:                          # FAISS pads with -1 when it finds < k
            continue
        hits.append(dict(score=float(score), **id_map[int(idx)]))
    return hits

show_hits("FAISS exact: 'container keeps dying out of memory'",
          faiss_search(flat_index, "container keeps dying out of memory", k=3))

FAISS exact: 'container keeps dying out of memory'
1. [+0.570] (ENG  | k8s_runbook.md)
     A container exits with OOMKilled when it exceeds its memory limit.
     Kubernetes terminates the process with exit code 137 and restarts the
     pod.
2. [+0.569] (ENG  | k8s_runbook.md)
     To remediate OOMKilled pods, raise resources.limits.memory in the
     deployment manifest, or profile the service for memory leaks before
     scaling.
3. [+0.343] (OPS  | incident_playbook.md)
     When the payment container crashes, first check the last deployment,
     then the pod events, then the upstream provider status page.



### 4.1 What FAISS actually returns

Run this once to internalise why the id map matters.

In [ ]:
q = embed(["container keeps dying out of memory"])
scores, ids = flat_index.search(q, 3)
print("raw scores :", scores)
print("raw ids    :", ids, "  <- just integers. No text. No source. No dept.")
print()
print("with our map:", [ID_MAP[i]["id"] for i in ids[0]])

raw scores : [[0.56988204 0.56863964 0.3432575 ]]
raw ids    : [[14 15 25]]   <- just integers. No text. No source. No dept.

with our map: ['en-03', 'en-04', 'op-02']


---
## 5. IVF — group and skip

36 vectors do not need an index. To feel IVF, PQ and HNSW we need scale, so we generate a
**synthetic 20,000-vector corpus** (clustered, like real embeddings — not uniform noise).
The AskCorp text corpus comes back in Section 8.

**IVF recap:**
1. `train()` runs k-means to find `nlist` centroids.
2. Each vector is assigned to its nearest centroid — an *inverted list* (bucket).
3. At search time, compare the query to the `nlist` centroids (**coarse search**), then
   scan only the vectors inside the closest `nprobe` buckets (**fine search**).
4. Cost drops from `O(N)` to roughly `O(N · nprobe / nlist)`.

In [ ]:
from sklearn.datasets import make_blobs

N, D, LATENT = 20_000, 128, 16

# Real embeddings are NOT random noise: they cluster by topic, and their dimensions are
# correlated (they live on a low-dimensional manifold). We reproduce both properties by
# making clusters in 16 dims and projecting them up to 128.
Z, _ = make_blobs(n_samples=N, n_features=LATENT, centers=60,
                  cluster_std=1.0, random_state=42)
W = np.random.RandomState(0).normal(size=(LATENT, D)) / np.sqrt(LATENT)
BIG = np.ascontiguousarray((Z @ W).astype("float32"))

# 100 held-out queries = real points nudged slightly, so a true neighbour exists.
Q = np.ascontiguousarray(
    (BIG[:100] + np.random.RandomState(1).normal(0, 0.3, (100, D))).astype("float32"))

show("Synthetic corpus for the index experiments", [
    f"vectors   : {N:,}  x  {D} dims (float32)",
    f"raw size  : {BIG.nbytes / 1e6:.1f} MB",
    f"queries   : {Q.shape[0]}",
])

Synthetic corpus for the index experiments
vectors   : 20,000  x  128 dims (float32)
raw size  : 10.2 MB
queries   : 100



In [ ]:
# ---- ground truth: exact L2 search, used to measure everyone else's recall ----
gt_index = faiss.IndexFlatL2(D)
gt_index.add(BIG)

K = 10
_, GT_IDS = gt_index.search(Q, K)

def recall_at_k(approx_ids, k=K, truth=GT_IDS):
    """Fraction of the true k nearest neighbours that the approximate index found."""
    hits = sum(len(set(a[:k]) & set(t[:k])) for a, t in zip(approx_ids, truth))
    return hits / (len(truth) * k)

def benchmark(index, name, nprobe=None, k=K):
    if nprobe is not None:
        index.nprobe = nprobe
    t0 = time.perf_counter()
    _, ids = index.search(Q, k)
    ms = (time.perf_counter() - t0) * 1000 / len(Q)
    size_mb = len(faiss.serialize_index(index)) / 1e6
    return dict(name=name, nprobe=nprobe, recall=recall_at_k(ids, k),
                ms_per_query=ms, size_mb=size_mb)

base = benchmark(gt_index, "IndexFlatL2 (exact)")
show("Baseline", [f"{base['name']}: recall={base['recall']:.3f}  "
                  f"{base['ms_per_query']:.2f} ms/query  {base['size_mb']:.1f} MB"])

Baseline
IndexFlatL2 (exact): recall=1.000  0.88 ms/query  10.2 MB



In [ ]:
def build_ivf(vecs, nlist=None):
    """IVF index. nlist ~ sqrt(N) is the rule of thumb from the lecture."""
    d = vecs.shape[1]
    nlist = nlist or int(np.sqrt(len(vecs)))
    quantizer = faiss.IndexFlatL2(d)                 # used to find the centroids
    index = faiss.IndexIVFFlat(quantizer, d, nlist)
    print(f"before train -> is_trained={index.is_trained}")
    index.train(vecs)                                # k-means happens here
    print(f"after  train -> is_trained={index.is_trained}, nlist={index.nlist}")
    index.add(vecs)
    return index

ivf = build_ivf(BIG)
print("vectors indexed:", ivf.ntotal)

before train -> is_trained=False
after  train -> is_trained=True, nlist=141
vectors indexed: 20000


### 5.1 `nprobe` is the recall dial

`nprobe = 1` is the fastest and least accurate. Raising it trades latency for recall until,
at `nprobe = nlist`, IVF degenerates back into exact search.

In [ ]:
rows = [f"{'nprobe':>7} | {'recall@10':>9} | {'ms/query':>9} | speedup vs exact",
        "-" * 56]
for nprobe in [1, 2, 4, 8, 16, 32, ivf.nlist]:
    r = benchmark(ivf, "IVF", nprobe=nprobe)
    rows.append(f"{nprobe:>7} | {r['recall']:>9.3f} | {r['ms_per_query']:>9.2f} | "
                f"{base['ms_per_query'] / r['ms_per_query']:.1f}x")
show("IVF: nprobe sweep", rows)

IVF: nprobe sweep
 nprobe | recall@10 |  ms/query | speedup vs exact
--------------------------------------------------------
      1 |     0.796 |      0.03 | 28.0x
      2 |     0.959 |      0.04 | 24.3x
      4 |     0.999 |      0.08 | 10.4x
      8 |     1.000 |      0.10 | 8.9x
     16 |     1.000 |      0.15 | 5.7x
     32 |     1.000 |      0.27 | 3.3x
    141 |     1.000 |      1.45 | 0.6x



> **Read the table, not the code.** Recall climbs steeply for the first few probes and then
> flattens — most of the accuracy comes from the first handful of buckets. That is the whole
> argument for IVF: skip ~90% of the comparisons and lose almost nothing.

### 5.2 The training-data warning

The lecture flags it: k-means needs roughly **30–40 points per centroid** or the clustering
is poor. Ask for too many clusters on too little data and FAISS warns you.

In [ ]:
tiny = BIG[:500]                       # 500 points...
bad = faiss.IndexIVFFlat(faiss.IndexFlatL2(D), D, 256)   # ...and 256 clusters
bad.train(tiny)                        # watch the WARNING that FAISS prints
print("\nRule of thumb: n_train >= ~40 * nlist  ->",
      f"256 clusters wants ~{40 * 256:,} points, we gave it {len(tiny)}.")


Rule of thumb: n_train >= ~40 * nlist  -> 256 clusters wants ~10,240 points, we gave it 500.


---
## 6. Product Quantization — compress

IVF solved *"too many vectors to check"*. PQ solves *"vectors are too big to store"*.

**The three steps from the slides:**
1. **Split** each `D`-dim vector into `m` sub-vectors.
2. **Cluster** each segment with k-means, `K = 256` centres → one codebook per segment.
3. **Encode** each vector as `m` centroid ids. `K = 256` means each id fits in **one byte**,
   so a vector costs `m` bytes regardless of `D`.

At query time, **ADC** (Asymmetric Distance Computation) keeps the query at full precision,
builds an `m × 256` lookup table once, and then scores each database vector with
`m` lookups and `m − 1` additions.

In [ ]:
def pq_memory_report(n, d, m, nbits=8):
    """The arithmetic behind the '384x smaller' slide."""
    raw   = n * d * 4                       # float32
    coded = n * m                           # nbits=8 -> 1 byte per segment
    books = (2 ** nbits) * d * 4            # the codebooks, stored once
    return dict(
        raw_MB=raw / 1e6,
        coded_MB=coded / 1e6,
        codebook_MB=books / 1e6,
        per_vector_raw=f"{d * 4} B",
        per_vector_coded=f"{m} B",
        compression=f"{raw / coded:.0f}x",
        lookup_table_entries=m * 2 ** nbits,
        ops_per_vector=f"{m} lookups + {m - 1} additions",
    )

show("PQ maths for our 20k corpus (D=128, m=8)",
     [f"{k:>22} : {v}" for k, v in pq_memory_report(N, D, 8).items()])

show("PQ maths at the slide's scale (N=1M, D=768, m=8)",
     [f"{k:>22} : {v}" for k, v in pq_memory_report(1_000_000, 768, 8).items()])

PQ maths for our 20k corpus (D=128, m=8)
                raw_MB : 10.24
              coded_MB : 0.16
           codebook_MB : 0.131072
        per_vector_raw : 512 B
      per_vector_coded : 8 B
           compression : 64x
  lookup_table_entries : 2048
        ops_per_vector : 8 lookups + 7 additions

PQ maths at the slide's scale (N=1M, D=768, m=8)
                raw_MB : 3072.0
              coded_MB : 8.0
           codebook_MB : 0.786432
        per_vector_raw : 3072 B
      per_vector_coded : 8 B
           compression : 384x
  lookup_table_entries : 2048
        ops_per_vector : 8 lookups + 7 additions



> **Sanity check the quiz answers from the lecture** with the function above:
> `pq_memory_report(10_000_000, 1024, m=16)` → the lookup table is `m × k = 4096` entries,
> and scoring one vector costs `16 lookups + 15 additions`.

In [ ]:
def build_pq(vecs, m=8, nbits=8):
    """Pure PQ: compresses every vector, but still scores all of them."""
    d = vecs.shape[1]
    assert d % m == 0, "d must be divisible by m"
    index = faiss.IndexPQ(d, m, nbits)
    index.train(vecs)          # learns m codebooks of 2**nbits centres
    index.add(vecs)            # stores only the m byte-ids per vector
    return index

def build_ivfpq(vecs, nlist=None, m=8, nbits=8):
    """IVF+PQ: cluster first, then PQ-encode the RESIDUAL (vector - centroid)."""
    d = vecs.shape[1]
    nlist = nlist or int(np.sqrt(len(vecs)))
    quantizer = faiss.IndexFlatL2(d)
    index = faiss.IndexIVFPQ(quantizer, d, nlist, m, nbits)
    index.train(vecs)
    index.add(vecs)
    return index

pq    = build_pq(BIG, m=8)
ivfpq = build_ivfpq(BIG, m=8)
print("built:", pq.ntotal, "and", ivfpq.ntotal, "vectors")

built: 20000 and 20000 vectors


In [ ]:
results = [
    base,
    benchmark(ivf,   "IVFFlat",  nprobe=8),
    benchmark(pq,    "PQ (m=8)"),
    benchmark(ivfpq, "IVF+PQ",   nprobe=8),
]

rows = [f"{'index':<20} | {'recall@10':>9} | {'ms/query':>9} | {'index MB':>9}", "-" * 58]
for r in results:
    rows.append(f"{r['name']:<20} | {r['recall']:>9.3f} | "
                f"{r['ms_per_query']:>9.2f} | {r['size_mb']:>9.1f}")
show("Head-to-head on the same 20k corpus", rows)

Head-to-head on the same 20k corpus
index                | recall@10 |  ms/query |  index MB
----------------------------------------------------------
IndexFlatL2 (exact)  |     1.000 |      0.88 |      10.2
IVFFlat              |     1.000 |      0.08 |      10.5
PQ (m=8)             |     0.371 |      1.85 |       0.3
IVF+PQ               |     0.669 |      0.21 |       0.5



Read that table carefully — it is the summary of the whole section:

* **IVFFlat** keeps the vectors intact, so recall stays near 1.0 and the index is the *same
  size* as the raw data. It only bought you **speed**.
* **PQ** shrinks the index by ~30x, and you can see the accuracy that costs. It only bought
  you **memory**.
* **IVF+PQ** is small *and* fast, and recovers a large part of PQ's lost recall — because it
  encodes residuals rather than raw vectors.

More segments buys back accuracy, at a proportional cost in bytes:

In [ ]:
rows = [f"{'m (segments)':<14}{'bytes/vector':<15}{'recall@10':<12}{'index MB'}", "-" * 52]
for m in [4, 8, 16, 32]:
    idx = build_pq(BIG, m=m)
    r = benchmark(idx, f"PQ m={m}")
    rows.append(f"{m:<14}{m:<15}{r['recall']:<12.3f}{r['size_mb']:.2f}")
rows += ["", f"(uncompressed for reference: {D * 4} bytes/vector, "
             f"recall 1.000, {base['size_mb']:.1f} MB)"]
show("PQ: the m dial", rows)

PQ: the m dial
m (segments)  bytes/vector   recall@10   index MB
----------------------------------------------------
4             4              0.285       0.21
8             8              0.371       0.29
16            16             0.452       0.45
32            32             0.621       0.77

(uncompressed for reference: 512 bytes/vector, recall 1.000, 10.2 MB)



### 6.1 Why residuals?

Plain PQ compresses the vector itself. IVF+PQ compresses `r = x − c`, the **offset from
the bucket centroid** — a short vector clustered near zero. Same number of bits, far finer
detail, because no bits are wasted re-describing what the centroid already told us.

*(The postal-address analogy from the slides: the city places you roughly, the street-level
offset does the rest.)*

In [ ]:
# Show that residuals really are smaller than the original vectors.
centroids = ivf.quantizer.reconstruct_n(0, ivf.nlist)
_, assign = ivf.quantizer.search(BIG[:2000], 1)
residuals = BIG[:2000] - centroids[assign[:, 0]]

show("Vector vs residual magnitude (2000 samples)", [
    f"mean ||x||  (original vector) : {np.linalg.norm(BIG[:2000], axis=1).mean():.2f}",
    f"mean ||r||  (residual)        : {np.linalg.norm(residuals, axis=1).mean():.2f}",
    "-> the residual is the shorter thing to encode, so the same 8 bytes buy more precision.",
])

Vector vs residual magnitude (2000 samples)
mean ||x||  (original vector) : 65.98
mean ||r||  (residual)        : 10.37
-> the residual is the shorter thing to encode, so the same 8 bytes buy more precision.



---
## 7. HNSW — the shortcut graph

The third trick from lecture. Link every vector to its nearest neighbours, then **hop**
towards the query — a few jumps instead of a full scan. No training required, excellent
recall, but the graph costs memory. This is the index ChromaDB uses under the hood.

In [ ]:
def build_hnsw(vecs, M=8, ef_construction=40):
    """M = neighbours per node. Production values are 16-64; we use 8 here so that
    efSearch visibly moves the recall needle on a corpus this small."""
    index = faiss.IndexHNSWFlat(vecs.shape[1], M)
    index.hnsw.efConstruction = ef_construction
    index.add(vecs)                     # note: no train() call
    return index

hnsw = build_hnsw(BIG, M=8)

rows = [f"{'efSearch':>9} | {'recall@10':>9} | {'ms/query':>9}", "-" * 34]
for ef in [4, 8, 16, 32, 64]:
    hnsw.hnsw.efSearch = ef
    r = benchmark(hnsw, "HNSW")
    rows.append(f"{ef:>9} | {r['recall']:>9.3f} | {r['ms_per_query']:>9.2f}")
rows += ["", f"index size: {len(faiss.serialize_index(hnsw)) / 1e6:.1f} MB "
             f"(vs {base['size_mb']:.1f} MB exact, {results[3]['size_mb']:.1f} MB IVF+PQ)"]
show("HNSW: efSearch is its recall dial", rows)

HNSW: efSearch is its recall dial
 efSearch | recall@10 |  ms/query
----------------------------------
        4 |     0.867 |      0.06
        8 |     0.948 |      0.04
       16 |     0.979 |      0.04
       32 |     0.991 |      0.07
       64 |     0.995 |      0.09

index size: 11.9 MB (vs 10.2 MB exact, 0.5 MB IVF+PQ)



### Section 5–7 takeaway

| Need | Index | Knobs |
|---|---|---|
| ≤ ~1M vectors, or 100% recall required | `IndexFlatIP` / `IndexFlatL2` | none |
| Millions, fits in RAM, small recall loss OK | `IndexIVFFlat` | `nlist ≈ √N`, `nprobe` |
| Too big for RAM | `IndexPQ` | `m`, `nbits` |
| Both problems at once | `IndexIVFPQ` | `nlist`, `nprobe`, `m` |
| Best recall/latency, memory to spare | `IndexHNSWFlat` | `M`, `efSearch` |

**You never implement these — you pick an index by name and the library does the rest.**

---
## 8. ChromaDB — a database wrapped around vector search

Back to the AskCorp corpus. FAISS gave us ids; Chroma stores **vector + text + metadata**
together, persists to disk, and — the thing raw ANN search cannot do — **filters on
metadata**.

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="./askcorp_chroma")   # persistence: it is on disk

if "askcorp" in [c.name for c in client.list_collections()]:
    client.delete_collection("askcorp")

collection = client.create_collection(
    name="askcorp",
    metadata={"hnsw:space": "cosine"},        # Chroma runs HNSW underneath
)

def index_corpus(collection, corpus, vecs):
    """Store all three things together: vector, document text, metadata."""
    collection.add(
        ids=[d["id"] for d in corpus],
        embeddings=[v.tolist() for v in vecs],
        documents=[d["text"] for d in corpus],
        metadatas=[{"dept": d["dept"], "source": d["source"], "year": d["year"]}
                   for d in corpus],
    )
    return collection.count()

print("stored:", index_corpus(collection, CORPUS, DOC_VECS), "records")

stored: 36 records


In [ ]:
def chroma_search(query, k=3, where=None):
    """Semantic search with an optional metadata pre-filter."""
    res = collection.query(
        query_embeddings=embed([query]).tolist(),
        n_results=k,
        where=where,                    # <-- the pre-filter
    )
    hits = []
    for doc_id, doc, meta, dist in zip(res["ids"][0], res["documents"][0],
                                       res["metadatas"][0], res["distances"][0]):
        hits.append(dict(id=doc_id, text=doc, score=1 - dist,
                         dept=meta["dept"], source=meta["source"]))
    return hits

show_hits("No filter — searches the whole company",
          chroma_search("What is the policy on leave?", k=3))

No filter — searches the whole company
1. [+0.489] (HR   | hr_policy.pdf)
     Contract and temporary staff are not covered by the standard parental
     leave policy. Their entitlements are defined individually in the
     vendor agreement.
2. [+0.473] (HR   | hr_policy.pdf)
     Full-time employees accrue 24 days of paid annual leave per calendar
     year. Unused leave up to 10 days may be carried into the next year.
3. [+0.447] (HR   | hr_policy.pdf)
     The hybrid work policy requires employees to be present in the office
     for a minimum of ten days per month, coordinated with their manager.



### 8.1 Metadata filtering — the superpower

Real questions are **scoped**: *"In the HR policy, how many sick days…"*. Turn that scope
into a `where` clause and you get higher precision, a smaller candidate set, and easier
citation — all from one line.

In [ ]:
query = "How is a crash investigated?"

show_hits("Unfiltered", chroma_search(query, k=3))
show_hits("where={'dept': 'OPS'}", chroma_search(query, k=3, where={"dept": "OPS"}))
show_hits("where={'source': 'k8s_runbook.md'}",
          chroma_search(query, k=3, where={"source": "k8s_runbook.md"}))

Unfiltered
1. [+0.352] (OPS  | oncall.md)
     Every production incident requires a blameless postmortem published
     within five working days of resolution.
2. [+0.271] (OPS  | incident_playbook.md)
     When the payment container crashes, first check the last deployment,
     then the pod events, then the upstream provider status page.
3. [+0.228] (OPS  | incident_playbook.md)
     Severity 1 incidents require an incident commander within five
     minutes and a status page update within fifteen minutes of
     declaration.

where={'dept': 'OPS'}
1. [+0.352] (OPS  | oncall.md)
     Every production incident requires a blameless postmortem published
     within five working days of resolution.
2. [+0.271] (OPS  | incident_playbook.md)
     When the payment container crashes, first check the last deployment,
     then the pod events, then the upstream provider status page.
3. [+0.228] (OPS  | incident_playbook.md)
     Severity 1 incidents require an incident commander within five
  

In [ ]:
# Filters compose with operators too.
show_hits("Only the current handbook: dept=HR AND year=2024",
          chroma_search("What is the insurance cover?", k=2,
                        where={"$and": [{"dept": "HR"}, {"year": 2024}]}))

show_hits("Only the OLD handbook: year=2023",
          chroma_search("What is the insurance cover?", k=2,
                        where={"year": 2023}))

Only the current handbook: dept=HR AND year=2024
1. [+0.526] (HR   | benefits_handbook.pdf)
     Health insurance covers the employee, spouse and two dependent
     children up to a sum insured of eight lakh rupees per year.
2. [+0.276] (HR   | hr_policy.pdf)
     Contract and temporary staff are not covered by the standard parental
     leave policy. Their entitlements are defined individually in the
     vendor agreement.

Only the OLD handbook: year=2023
1. [+0.471] (HR   | benefits_handbook.pdf)
     In the 2023 handbook the health insurance sum insured was five lakh
     rupees. This figure was revised upward in the 2024 revision.



> Same question, two different correct answers — because the corpus contains both the 2023
> and 2024 figures. Without metadata there is no way to ask for "the current one".

### 8.2 Pre-filter vs post-filter

* **Pre-filter** — narrow first, then search only the matches. Faster, more precise,
  reliably fills `k`. Needs an index that supports it.
* **Post-filter** — search everything, then drop non-matches. Works on any index, but may
  return **fewer than `k`** results and wastes work. Fix: over-fetch, then filter.

In [ ]:
def post_filter_search(query, k=3, dept=None, fetch=None):
    """Search everything first, then discard. fetch=k is the naive (broken) version."""
    fetch = fetch or k
    hits = chroma_search(query, k=fetch)                 # no where= clause
    kept = [h for h in hits if dept is None or h["dept"] == dept]
    return kept[:k], len(hits)

q = "What is the policy on leave?"

pre = chroma_search(q, k=3, where={"dept": "OPS"})
naive, scanned = post_filter_search(q, k=3, dept="OPS")
over,  scanned2 = post_filter_search(q, k=3, dept="OPS", fetch=20)

show("Asking an HR-flavoured question but restricting to OPS", [
    f"pre-filter                : returned {len(pre)}/3   (index searched only OPS docs)",
    f"post-filter, fetch=3      : returned {len(naive)}/3   <- came up SHORT",
    f"post-filter, fetch=20     : returned {len(over)}/3   <- fixed by over-fetching,",
    f"                            but it scanned {scanned2} rows to keep {len(over)}.",
])

Asking an HR-flavoured question but restricting to OPS
pre-filter                : returned 3/3   (index searched only OPS docs)
post-filter, fetch=3      : returned 0/3   <- came up SHORT
post-filter, fetch=20     : returned 3/3   <- fixed by over-fetching,
                            but it scanned 20 rows to keep 3.



> **Rule of thumb from the lecture:** prefer pre-filtering whenever your vector DB supports
> it. Post-filtering is the fallback, and it needs over-fetching to be safe.

### 8.3 Persistence — the other thing FAISS does not give you

In [ ]:
del client, collection
client2 = chromadb.PersistentClient(path="./askcorp_chroma")
reopened = client2.get_collection("askcorp")
print("reopened from disk with", reopened.count(), "records — no re-embedding needed")

collection = reopened          # keep using it below

reopened from disk with 36 records — no re-embedding needed


---
## 9. Reranking — the two-stage pattern

Stage 1 (bi-encoder + ANN) optimises **recall, cheaply**: pull ~50 candidates from millions.
Stage 2 (cross-encoder) optimises **precision, expensively**: re-score just those candidates.
Then the best 5 go to the LLM.

The **cross-encoder** reads query and document **together** in one model, so it sees
word-by-word interaction. Far more accurate — and far too slow to run on the whole corpus.
That cost asymmetry *is* the reason for two stages.

In [ ]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidates, top_n=3):
    """Re-score (query, document) pairs jointly and reorder."""
    pairs  = [(query, c["text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)
    out = [dict(c, score=float(s), retriever_rank=i + 1)
           for i, (c, s) in enumerate(zip(candidates, scores))]
    out.sort(key=lambda h: -h["score"])
    return out[:top_n]

def retrieve_then_rerank(query, fetch_k=15, top_n=3, where=None):
    stage1 = chroma_search(query, k=fetch_k, where=where)   # fast, approximate
    stage2 = rerank(query, stage1, top_n=top_n)             # slow, precise
    return stage1, stage2

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
q = "Why would my access token stop working sooner than expected?"
stage1, stage2 = retrieve_then_rerank(q, fetch_k=15, top_n=3)

show_hits("STAGE 1 — bi-encoder retrieval (top 5 of 15)", stage1[:5])
show_hits("STAGE 2 — cross-encoder rerank (top 3)", stage2)

show("Movement caused by the reranker",
     [f"final #{i}  <-  retriever rank #{h['retriever_rank']}"
      for i, h in enumerate(stage2, 1)])

STAGE 1 — bi-encoder retrieval (top 5 of 15)
1. [+0.599] (ENG  | auth_service.md)
     The auth module issues JWT access tokens with a 15 minute expiry and
     refresh tokens valid for 24 hours. Token lifetime is set by
     AUTH_TOKEN_TTL.
2. [+0.560] (ENG  | auth_service.md)
     If clients report tokens expiring early, check for clock skew between
     the auth pods and the API gateway; skew above 60 seconds invalidates
     tokens prematurely.
3. [+0.361] (ENG  | api_guide.md)
     The public API is rate limited to 1000 requests per minute per API
     key. Exceeding the limit returns HTTP 429 with a Retry-After header.
4. [+0.276] (OPS  | network_manual.pdf)
     Network error code NET-503 means the access point lost its uplink to
     the distribution switch and fell back to mesh mode.
5. [+0.267] (HR   | hr_policy.pdf)
     Employees may request a sabbatical of up to six months after five
     years of continuous service, subject to business approval.

STAGE 2 — cross-encoder r

### 9.1 The cost asymmetry, measured

This is the number that justifies the whole architecture.

In [ ]:
def timing_comparison(query, n_docs):
    docs = (DOCS * 100)[:n_docs]

    t0 = time.perf_counter()
    embed([query]) @ DOC_VECS.T                     # bi-encoder: 1 encode + a dot product
    bi_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    cross_encoder.predict([(query, d) for d in docs])   # cross-encoder: n forward passes
    cr_ms = (time.perf_counter() - t0) * 1000
    return bi_ms, cr_ms

bi_ms, cr_ms = timing_comparison("token expiry", 100)
show("Scoring 100 documents", [
    f"bi-encoder    : {bi_ms:7.1f} ms   (doc vectors were precomputed offline)",
    f"cross-encoder : {cr_ms:7.1f} ms   ({cr_ms / bi_ms:.0f}x slower)",
    "",
    f"Extrapolated to 1,000,000 docs: cross-encoder ~= {cr_ms * 10_000 / 1000 / 60:.0f} minutes"
    " for ONE query.",
    "-> which is exactly why stage 1 narrows to ~100 first.",
])

Scoring 100 documents
bi-encoder    :    22.3 ms   (doc vectors were precomputed offline)
cross-encoder :  1754.9 ms   (79x slower)

Extrapolated to 1,000,000 docs: cross-encoder ~= 292 minutes for ONE query.
-> which is exactly why stage 1 narrows to ~100 first.



---
## 10. Request shaping — query transformations

> *"Requests made by users are not always precise and usable."*

Everything so far assumed a good query. Section 10 fixes the query **before** retrieval.

### The LLM helper

The lab needs an LLM for these steps. It runs three ways — pick one and move on:

* **Offline stub (default).** Deterministic, rule-based, no key, no network. Clearly marked.
* **OpenAI-compatible API.** Set `OPENAI_API_KEY` below.
* **Local model.** Swap `call_llm` for any `transformers` pipeline.

The *point* of this section is the prompt patterns and their effect on retrieval — not the
model. The stub is enough to see the effect.

In [ ]:
OPENAI_API_KEY = ""          # <- paste a key to use a real LLM; leave blank for the stub
LLM_MODEL      = "gpt-4o-mini"

def call_llm(prompt, task=None, **ctx):
    """Single entry point for every LLM call in this notebook."""
    if OPENAI_API_KEY:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        r = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        return r.choices[0].message.content.strip()
    return _stub_llm(task, **ctx)


GLOSSARY = {
    "oomkilled": "a container being terminated for exceeding its memory limit in Kubernetes",
    "e-4021":    "the API error code for a malformed pagination cursor",
    "net-503":   "the network error code for an access point losing its uplink",
    "pto":       "paid time off / annual leave",
    "429":       "an HTTP rate limit response",
}

def _stub_llm(task, **ctx):
    """Deterministic stand-in for an LLM. Rule-based on purpose, so results are reproducible."""
    q = ctx.get("query", "")
    if task == "rewrite":
        meaning = GLOSSARY.get(q.strip().lower())
        if meaning:
            return f"What causes {q.strip()} ({meaning}), and what are the remediation steps?"
        return f"What are the company's documented details and procedures regarding: {q.rstrip('?.')}?"
    if task == "standalone":
        topic = ctx.get("topic", "the previous topic")
        return re.sub(r"\b(it|this|that|they|them)\b", topic, q, flags=re.I)
    if task == "multiquery":
        m = re.search(r"both\s+(.+?)\s+and\s+(.+?)([?.]|$)", q, re.I)
        if m:            # rebuild the whole sentence once per entity -- keeps it readable
            subs = [q[:m.start()] + e + q[m.end(2):] for e in (m.group(1), m.group(2))]
        elif ctx.get("entities"):
            subs = [f"{q.rstrip('?')} for {e}?" for e in ctx["entities"]]
        else:
            subs = [q]
        return "\n".join(f"{i}. {s}" for i, s in enumerate(subs, 1))
    if task == "stepback":
        return ("What is the overall architecture and set of rules governing "
                f"{ctx.get('subject', 'this system')}, and what data does it collect?")
    if task == "route":
        return ctx.get("fallback_route", "GENERAL")
    if task == "generate":
        ctxt = ctx.get("chunks", [])
        if not ctxt:
            return "I could not find anything in the knowledge base about that."
        lead = ctxt[0]
        cites = ", ".join(sorted({c["source"] for c in ctxt}))
        return f"{lead['text']}\n\n(Sources: {cites})"
    return "[stub llm: no task matched]"

print("LLM mode:", "REAL API" if OPENAI_API_KEY else "offline stub (deterministic)")

LLM mode: offline stub (deterministic)


### 10.1 Rewriting — vague or jargon queries → specific search statements

Essential for Slack/Teams style channels where people paste fragments and error codes.

In [ ]:
def rewrite_query(query):
    prompt = f"""Rewrite the following user query into a clear and specific search question.
User query: {query}
Return only the rewritten question."""
    return call_llm(prompt, task="rewrite", query=query)

for raw in ["OOMKilled", "car leasing policy", "E-4021"]:
    print(f"  raw : {raw}")
    print(f"  new : {rewrite_query(raw)}\n")

  raw : OOMKilled
  new : What causes OOMKilled (a container being terminated for exceeding its memory limit in Kubernetes), and what are the remediation steps?

  raw : car leasing policy
  new : What are the company's documented details and procedures regarding: car leasing policy?

  raw : E-4021
  new : What causes E-4021 (the API error code for a malformed pagination cursor), and what are the remediation steps?



In [ ]:
def compare_retrieval(original, transformed, k=3, where=None):
    """Did the transformation actually change what comes back?"""
    a = chroma_search(original,    k=k, where=where)
    b = chroma_search(transformed, k=k, where=where)
    rows = [f"{'':4}{'ORIGINAL':<34}{'TRANSFORMED':<34}", "-" * 72]
    for i in range(k):
        rows.append(f"{i+1:<4}{a[i]['id'] + ' ' + a[i]['source']:<34}"
                    f"{b[i]['id'] + ' ' + b[i]['source']:<34}")
    rows += ["", f"mean score  {np.mean([h['score'] for h in a]):.3f}"
                 f"                    {np.mean([h['score'] for h in b]):.3f}"]
    show(f"'{original}'  ->  '{transformed[:45]}...'", rows)

compare_retrieval("OOMKilled", rewrite_query("OOMKilled"))

'OOMKilled'  ->  'What causes OOMKilled (a container being term...'
    ORIGINAL                          TRANSFORMED                       
------------------------------------------------------------------------
1   en-03 k8s_runbook.md              en-03 k8s_runbook.md              
2   en-04 k8s_runbook.md              en-04 k8s_runbook.md              
3   op-09 oncall.md                   op-02 incident_playbook.md        

mean score  0.276                    0.691



### 10.2 Follow-up → standalone question

*"Does it apply to contractors?"* retrieves nothing useful, because **"it" carries no
meaning to embed**. Inject the conversation history and the query becomes self-contained.

In [ ]:
CONVERSATION = [
    ("user",  "What is the standard parental leave policy?"),
    ("agent", "The primary caregiver receives 26 weeks of paid leave."),
]

def make_standalone(query, history=CONVERSATION):
    hist = "\n".join(f"{r}: {t}" for r, t in history)
    prompt = f"""Rewrite the user's follow-up question into a complete, standalone question
using the conversation history.

Conversation history:
{hist}

Follow-up question:
{query}

Return only the rewritten question."""
    last_user = [t for r, t in history if r == "user"][-1]
    topic = last_user.lower().replace("what is the", "").replace("?", "").strip()
    return call_llm(prompt, task="standalone", query=query, topic=topic)

followup = "Does it apply to contractors?"
standalone = make_standalone(followup)
print("follow-up :", followup)
print("standalone:", standalone, "\n")

show_hits("Retrieval on the RAW follow-up", chroma_search(followup, k=2))
show_hits("Retrieval on the STANDALONE question", chroma_search(standalone, k=2))

follow-up : Does it apply to contractors?
standalone: Does standard parental leave policy apply to contractors? 

Retrieval on the RAW follow-up
1. [+0.288] (HR   | hr_policy.pdf)
     Contract and temporary staff are not covered by the standard parental
     leave policy. Their entitlements are defined individually in the
     vendor agreement.
2. [+0.264] (HR   | benefits_handbook.pdf)
     The employee car-leasing benefit allows staff at grade M3 and above
     to lease a vehicle through payroll deduction over a 36-month term.

Retrieval on the STANDALONE question
1. [+0.784] (HR   | hr_policy.pdf)
     Contract and temporary staff are not covered by the standard parental
     leave policy. Their entitlements are defined individually in the
     vendor agreement.
2. [+0.599] (HR   | hr_policy.pdf)
     The standard parental leave policy grants 26 weeks of paid leave to
     the primary caregiver and 4 weeks to the secondary caregiver.



> The raw follow-up is a coin flip; the standalone question pulls `hr-04` — the passage that
> actually says contractors are excluded. Same retriever, better query.

### 10.3 Multi-query retrieval — one complex question → several simple ones

When a question spans two entities, one embedding is a blurred average of both. Split it,
retrieve in parallel, then merge.

In [ ]:
def multi_query(query, entities=None):
    prompt = f"""Break the following question into separate, simple search queries.
Each query should focus on ONE entity.

User question:
{query}

Return the queries as a numbered list."""
    out = call_llm(prompt, task="multiquery", query=query, entities=entities)
    return [re.sub(r"^\s*\d+[.)]\s*", "", line).strip()
            for line in out.splitlines() if line.strip()]

def multi_query_retrieve(queries, k_each=2, final_k=4):
    """Retrieve for each sub-query, then merge with reciprocal-rank fusion."""
    fused = {}
    for q in queries:
        for rank, h in enumerate(chroma_search(q, k=k_each), 1):
            fused.setdefault(h["id"], dict(h, score=0.0))
            fused[h["id"]]["score"] += 1.0 / (60 + rank)     # RRF
    return sorted(fused.values(), key=lambda h: -h["score"])[:final_k]

complex_q = "Did the latency spike happen in both the Mumbai and US-East deployments?"
subs = multi_query(complex_q, entities=["the Mumbai deployment", "the US-East deployment"])

print("sub-queries:")
for s in subs:
    print("   -", s)
print()

show_hits("SINGLE query", chroma_search(complex_q, k=4))
show_hits("MULTI query + reciprocal rank fusion", multi_query_retrieve(subs))

sub-queries:
   - Did the latency spike happen in the Mumbai?
   - Did the latency spike happen in US-East deployments?

SINGLE query
1. [+0.686] (OPS  | monitoring.md)
     Latency dashboards are published per region. The Mumbai and US-East
     deployments are monitored on separate boards with independent alert
     thresholds.
2. [+0.606] (OPS  | monitoring.md)
     The Mumbai region reported a p99 latency spike of 2.4 seconds on 14
     March, traced to an undersized connection pool on the read replica.
3. [+0.451] (OPS  | monitoring.md)
     The US-East region remained within its 400 millisecond p99 budget
     through March and raised no latency alerts during that window.
4. [+0.242] (ENG  | db_schema.md)
     The orders table is partitioned by month. Queries that do not filter
     on order_date will scan every partition and are the usual cause of
     slow reports.

MULTI query + reciprocal rank fusion
1. [+0.033] (OPS  | monitoring.md)
     The Mumbai region reported a p99 lat

> **The hidden cost (lecture quiz):** N sub-queries means N retrievals and N chunk sets all
> competing for the same context window. Multi-query is not free — use it when the question
> genuinely spans entities.

### 10.4 Step-back prompting — retrieve the foundation first

When a question is so specific that no chunk matches it, ask a *broader* question first to
pull in the architectural rule that explains why.

In [ ]:
def step_back(query, subject=None):
    prompt = f"""The user's question is very specific.
Create a broader, high-level question that will help retrieve the basic information
needed to answer it.

User question:
{query}

Return only the broader question."""
    return call_llm(prompt, task="stepback", query=query,
                    subject=subject or "the network monitoring system")

specific = "Why can't I pull switch association data for the rogue units on Floor 3?"
broad = step_back(specific)

print("specific :", specific)
print("step-back:", broad, "\n")

show_hits("Retrieval on the SPECIFIC query", chroma_search(specific, k=2))
show_hits("Retrieval on the STEP-BACK query", chroma_search(broad, k=2))

specific : Why can't I pull switch association data for the rogue units on Floor 3?
step-back: What is the overall architecture and set of rules governing the network monitoring system, and what data does it collect? 

Retrieval on the SPECIFIC query
1. [+0.791] (OPS  | network_manual.pdf)
     Switch association data is collected only for reporting access
     points. Rogue units are detected by signal only, so no switch
     association data exists for them.
2. [+0.206] (ENG  | db_schema.md)
     Schema migrations run through Alembic in the release pipeline.
     Backward incompatible column drops must be split across two releases.

Retrieval on the STEP-BACK query
1. [+0.373] (OPS  | monitoring.md)
     Latency dashboards are published per region. The Mumbai and US-East
     deployments are monitored on separate boards with independent alert
     thresholds.
2. [+0.318] (ENG  | logging_guide.md)
     All services emit structured JSON logs with a trace_id field. Use the
     trace_id

> `op-10` contains the architectural rule: switch association data exists only for
> *reporting* access points. With it the agent can explain the limitation instead of
> hallucinating a fix.

### 10.5 All four in one table

In [ ]:
show("Choosing a transformation", [
    f"{'technique':<22}{'use when':<44}{'risk'}",
    "-" * 90,
    f"{'Rewriting':<22}{'jargon, fragments, error codes':<44}can drift from user intent",
    f"{'Follow-up rewrite':<22}{'multi-turn, pronouns (it/they/that)':<44}useless on turn 1",
    f"{'Multi-query':<22}{'question spans 2+ entities':<44}N retrievals, context bloat",
    f"{'Step-back':<22}{'hyper-specific, no results':<44}can retrieve too generic",
])

Choosing a transformation
technique             use when                                    risk
------------------------------------------------------------------------------------------
Rewriting             jargon, fragments, error codes              can drift from user intent
Follow-up rewrite     multi-turn, pronouns (it/they/that)         useless on turn 1
Multi-query           question spans 2+ entities                  N retrievals, context bloat
Step-back             hyper-specific, no results                  can retrieve too generic



---
## 11. Query routing — the traffic cop

We now have four transformations and three knowledge collections. **How does the system
decide which to use?** Hardcoded `if/else` for every scenario is brittle. Routing is the
layer that decides dynamically.

We will build all four strategies against the same three routes, then combine them into
the hybrid waterfall.

In [ ]:
ROUTES = ["HR", "ENG", "OPS"]

ROUTE_TEST_SET = [
    ("How many days of leave do I have left?",              "HR"),
    ("My salary was credited short this month",             "HR"),
    ("I'm feeling burnt out and need a break next Tuesday", "HR"),
    ("Why does my container keep restarting?",              "ENG"),
    ("How do I open a pull request?",                       "ENG"),
    ("The auth token is expiring too fast",                 "ENG"),
    ("Pipeline failed.",                                    "OPS"),
    ("Who is on call this week?",                           "OPS"),
    ("The payment container crashed five minutes ago",      "OPS"),
]

def evaluate_router(router, name, test_set=ROUTE_TEST_SET, verbose=True):
    rows, correct = [f"{'query':<52}{'pred':<6}{'gold':<6}{''}", "-" * 70], 0
    for q, gold in test_set:
        pred = router(q)
        ok = (pred == gold)
        correct += ok
        rows.append(f"{q[:50]:<52}{str(pred):<6}{gold:<6}{'OK' if ok else 'X'}")
    acc = correct / len(test_set)
    rows += ["", f"accuracy: {correct}/{len(test_set)} = {acc:.0%}"]
    if verbose:
        show(name, rows)
    return acc

### 11.1 Rule-based — deterministic, free, ~1 ms, and brittle

In [ ]:
RULES = {
    "HR":  ["leave", "pto", "payroll", "salary", "benefit", "insurance", "parental",
            "sick", "sabbatical", "reimburse"],
    "ENG": ["bug", "pull request", "pr ", "code", "api", "token", "container",
            "deploy", "schema", "python"],
    "OPS": ["pipeline", "incident", "server", "on call", "on-call", "outage",
            "latency", "postmortem", "crash"],
}

def rule_router(query, rules=RULES, default=None):
    q = query.lower()
    for route, keywords in rules.items():
        if any(kw in q for kw in keywords):
            return route
    return default

evaluate_router(lambda q: rule_router(q, default="?"), "Rule-based routing")

Rule-based routing
query                                               pred  gold  
----------------------------------------------------------------------
How many days of leave do I have left?              HR    HR    OK
My salary was credited short this month             HR    HR    OK
I'm feeling burnt out and need a break next Tuesda  ?     HR    X
Why does my container keep restarting?              ENG   ENG   OK
How do I open a pull request?                       ENG   ENG   OK
The auth token is expiring too fast                 ENG   ENG   OK
Pipeline failed.                                    OPS   OPS   OK
Who is on call this week?                           OPS   OPS   OK
The payment container crashed five minutes ago      ENG   OPS   X

accuracy: 7/9 = 78%



0.7777777777777778

In [ ]:
show("Where rules break", [
    f"'I need to take time off'          -> {rule_router('I need to take time off', default='NO MATCH')}"
     "   (no exact keyword)",
    f"'The payroll server crashed'       -> {rule_router('The payroll server crashed', default='NO MATCH')}"
     "   (matches HR first -- WRONG, this is OPS)",
    "",
    "Deterministic - Zero cost - ~1 ms latency - Low generalisation.",
])

Where rules break
'I need to take time off'          -> NO MATCH   (no exact keyword)
'The payroll server crashed'       -> HR   (matches HR first -- WRONG, this is OPS)

Deterministic - Zero cost - ~1 ms latency - Low generalisation.



> That second line is the lecture's quiz answer: a keyword match **short-circuits the
> waterfall**, so the smarter stages never run. Rules can be confidently wrong.

**The fix that makes rules safe to put first:** only accept the rule stage when exactly
*one* route matches. If two routes both fire, the query is ambiguous — abstain and let a
smarter stage decide. This is the version the hybrid router will use.

In [ ]:
def rule_router_strict(query, rules=RULES):
    """Return a route only if exactly ONE route matches. Ambiguity -> abstain (None)."""
    q = query.lower()
    matched = [r for r, kws in rules.items() if any(kw in q for kw in kws)]
    return matched[0] if len(matched) == 1 else None

for q in ["How many days of leave do I have left?",
          "The payroll server crashed",
          "Did the latency spike happen in both the Mumbai and US-East deployments?"]:
    print(f"{q[:52]:<54} naive={str(rule_router(q)):<6} strict={rule_router_strict(q)}")

How many days of leave do I have left?                 naive=HR     strict=HR
The payroll server crashed                             naive=HR     strict=None
Did the latency spike happen in both the Mumbai and    naive=ENG    strict=None


> `"The payroll server crashed"` fires **HR** (payroll) *and* **OPS** (server, crash), so the
> strict version abstains instead of guessing. Abstention is a feature: the cheap stage is
> allowed to say *"I don't know"*.

### 11.2 Embedding-based — semantic, still no LLM

Write 3–5 sample questions per route (a *route profile*), embed them, and match the query
by cosine similarity. Fast, offline-capable, and it generalises past exact keywords.

In [ ]:
ROUTE_PROFILES = {
    "HR": [
        "How many days of leave do I have?",
        "What are my employee benefits?",
        "Can I take a vacation next week?",
        "My salary or payroll seems incorrect",
        "What is the parental leave policy?",
    ],
    "ENG": [
        "Why is my program crashing?",
        "How do I fix this bug in the service?",
        "How do I create a pull request?",
        "The API is returning an error code",
        "How does authentication work in our system?",
    ],
    "OPS": [
        "The deployment pipeline is failing",
        "A production server is down",
        "Who is the on-call engineer?",
        "There is a latency spike in production",
        "How do I handle a severity 1 incident?",
    ],
}

PROFILE_VECS = {r: embed(qs) for r, qs in ROUTE_PROFILES.items()}

def embedding_router(query, threshold=0.0, return_scores=False):
    qv = embed([query])[0]
    scores = {r: float((V @ qv).max()) for r, V in PROFILE_VECS.items()}
    best = max(scores, key=scores.get)
    if return_scores:
        return best, scores
    return best if scores[best] >= threshold else None

evaluate_router(embedding_router, "Embedding-based routing")

Embedding-based routing
query                                               pred  gold  
----------------------------------------------------------------------
How many days of leave do I have left?              HR    HR    OK
My salary was credited short this month             HR    HR    OK
I'm feeling burnt out and need a break next Tuesda  HR    HR    OK
Why does my container keep restarting?              ENG   ENG   OK
How do I open a pull request?                       ENG   ENG   OK
The auth token is expiring too fast                 ENG   ENG   OK
Pipeline failed.                                    OPS   OPS   OK
Who is on call this week?                           OPS   OPS   OK
The payment container crashed five minutes ago      OPS   OPS   OK

accuracy: 9/9 = 100%



1.0

In [ ]:
for q in ["I need to take time off", "The payroll server crashed", "Pipeline failed."]:
    best, scores = embedding_router(q, return_scores=True)
    print(f"{q:<35} -> {best:<4}  " +
          "  ".join(f"{r}={s:.2f}" for r, s in scores.items()))

I need to take time off             -> HR    HR=0.26  ENG=0.07  OPS=0.20
The payroll server crashed          -> OPS   HR=0.33  ENG=0.33  OPS=0.51
Pipeline failed.                    -> OPS   HR=0.14  ENG=0.25  OPS=0.72


> Note the last line: *"Pipeline failed."* scores low on **every** route. A low top score is
> the confidence signal the hybrid router will use to escalate.

### 11.3 Classifier-based — learned intent

Train a lightweight model on historical queries → labels. Gives calibrated probabilities.
The catch: it needs labelled history, which a brand-new product does not have.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

TRAIN_X, TRAIN_Y = [], []
for route, qs in ROUTE_PROFILES.items():
    TRAIN_X += qs + [d["text"] for d in CORPUS if d["dept"] == route]
    TRAIN_Y += [route] * (len(qs) + sum(1 for d in CORPUS if d["dept"] == route))

vectorizer = CountVectorizer(stop_words="english")
X = vectorizer.fit_transform(TRAIN_X)
classifier = MultinomialNB().fit(X, TRAIN_Y)

def classifier_router(query, return_proba=False):
    v = vectorizer.transform([query])
    pred = classifier.predict(v)[0]
    if return_proba:
        proba = dict(zip(classifier.classes_, classifier.predict_proba(v)[0]))
        return pred, proba
    return pred

print(f"trained on {len(TRAIN_X)} labelled examples\n")
evaluate_router(classifier_router, "Classifier-based routing (Naive Bayes)")

pred, proba = classifier_router("My compensation seems wrong this month", return_proba=True)
print("confidence:", {k: round(v, 3) for k, v in proba.items()})

trained on 51 labelled examples

Classifier-based routing (Naive Bayes)
query                                               pred  gold  
----------------------------------------------------------------------
How many days of leave do I have left?              HR    HR    OK
My salary was credited short this month             HR    HR    OK
I'm feeling burnt out and need a break next Tuesda  ENG   HR    X
Why does my container keep restarting?              OPS   ENG   X
How do I open a pull request?                       ENG   ENG   OK
The auth token is expiring too fast                 ENG   ENG   OK
Pipeline failed.                                    OPS   OPS   OK
Who is on call this week?                           HR    OPS   X
The payment container crashed five minutes ago      OPS   OPS   OK

accuracy: 6/9 = 67%

confidence: {np.str_('ENG'): np.float64(0.277), np.str_('HR'): np.float64(0.578), np.str_('OPS'): np.float64(0.145)}


### 11.4 LLM-based — accurate on ambiguity, but you pay for it

In [ ]:
ROUTE_DESCRIPTIONS = """- HR: employee policy, leave, payroll, benefits, insurance
- ENG: software, code, bugs, APIs, authentication, database schema
- OPS: servers, deployments, pipelines, incidents, monitoring, on-call"""

def llm_router(query):
    prompt = f"""You are a routing system. Choose the best route for the user's question.

Available routes:
{ROUTE_DESCRIPTIONS}

User question:
{query}

Return only the route name."""
    out = call_llm(prompt, task="route", query=query,
                   fallback_route=embedding_router(query))   # stub defers to embeddings
    out = out.strip().upper()
    return out if out in ROUTES else embedding_router(query)

evaluate_router(llm_router, "LLM-based routing")

LLM-based routing
query                                               pred  gold  
----------------------------------------------------------------------
How many days of leave do I have left?              HR    HR    OK
My salary was credited short this month             HR    HR    OK
I'm feeling burnt out and need a break next Tuesda  HR    HR    OK
Why does my container keep restarting?              ENG   ENG   OK
How do I open a pull request?                       ENG   ENG   OK
The auth token is expiring too fast                 ENG   ENG   OK
Pipeline failed.                                    OPS   OPS   OK
Who is on call this week?                           OPS   OPS   OK
The payment container crashed five minutes ago      OPS   OPS   OK

accuracy: 9/9 = 100%



1.0

> With `OPENAI_API_KEY` blank, the stub delegates to the embedding router so the notebook
> stays runnable. Paste a key and this cell becomes a genuine LLM classifier — the accuracy
> on ambiguous queries should go **up**, and the latency with it.

### 11.5 Hybrid — the waterfall

`Rule → Embedding → LLM`, escalating only on low confidence. You pay for the expensive
stage **only when the cheap ones fail**.

In [ ]:
def hybrid_router(query, embed_threshold=0.55, verbose=True):
    log = []

    # Stage 1: rules (strict -- abstains when two routes match)
    r = rule_router_strict(query)
    if r:
        log.append(f"Stage 1 rule-based   : SUCCESS -> {r}   (cost $0, ~1ms)")
        if verbose: show(f"Routing '{query}'", log)
        return r
    log.append("Stage 1 rule-based   : FAIL (no match, or ambiguous match)")

    # Stage 2: embeddings
    best, scores = embedding_router(query, return_scores=True)
    top = scores[best]
    if top >= embed_threshold:
        log.append(f"Stage 2 embedding    : SUCCESS -> {best} (score {top:.2f}), "
                   f"cost $0, ~10ms")
        if verbose: show(f"Routing '{query}'", log)
        return best
    log.append(f"Stage 2 embedding    : LOW CONFIDENCE (best {best} @ {top:.2f} "
               f"< {embed_threshold})")

    # Stage 3: LLM
    r = llm_router(query)
    log.append(f"Stage 3 LLM          : SUCCESS -> {r}   (costs money, ~500ms)")
    if verbose: show(f"Routing '{query}'", log)
    return r

hybrid_router("How many days of leave do I have left?")
hybrid_router("I'm feeling burnt out and need a break next Tuesday")
hybrid_router("Pipeline failed.")

Routing 'How many days of leave do I have left?'
Stage 1 rule-based   : SUCCESS -> HR   (cost $0, ~1ms)

Routing 'I'm feeling burnt out and need a break next Tuesday'
Stage 1 rule-based   : FAIL (no match, or ambiguous match)
Stage 2 embedding    : LOW CONFIDENCE (best HR @ 0.42 < 0.55)
Stage 3 LLM          : SUCCESS -> HR   (costs money, ~500ms)

Routing 'Pipeline failed.'
Stage 1 rule-based   : SUCCESS -> OPS   (cost $0, ~1ms)



'OPS'

In [ ]:
def acc(router):
    return f"{evaluate_router(router, '', verbose=False):.0%}"

show("Strategy comparison", [
    f"{'strategy':<14}{'accuracy':<11}{'latency':<11}{'cost':<8}{'needs'}",
    "-" * 74,
    f"{'Rule':<14}{acc(rule_router):<11}{'~1 ms':<11}{'$0':<8}hand-written keywords",
    f"{'Embedding':<14}{acc(embedding_router):<11}{'~10 ms':<11}{'$0':<8}a few sample questions",
    f"{'Classifier':<14}{acc(classifier_router):<11}{'~1 ms':<11}{'$0':<8}LABELLED HISTORY",
    f"{'LLM':<14}{acc(llm_router):<11}{'~500 ms':<11}{'$$':<8}an API key",
    f"{'Hybrid':<14}{acc(lambda q: hybrid_router(q, verbose=False)):<11}"
    f"{'adaptive':<11}{'$':<8}all of the above",
    "",
    "Quiz: a brand-new startup with zero query logs cannot deploy the CLASSIFIER --",
    "it needs labelled historical queries. Route profiles can just be hand-written.",
])

Strategy comparison
strategy      accuracy   latency    cost    needs
--------------------------------------------------------------------------
Rule          78%        ~1 ms      $0      hand-written keywords
Embedding     100%       ~10 ms     $0      a few sample questions
Classifier    67%        ~1 ms      $0      LABELLED HISTORY
LLM           100%       ~500 ms    $$      an API key
Hybrid        100%       adaptive   $       all of the above

Quiz: a brand-new startup with zero query logs cannot deploy the CLASSIFIER --
it needs labelled historical queries. Route profiles can just be hand-written.



---
## 12. The full AskCorp pipeline

Every piece, in order:

```
query
  -> TRANSFORM        (rewrite / standalone / multi-query / step-back)
  -> ROUTE            (hybrid: rule -> embedding -> LLM)
  -> RETRIEVE         (ChromaDB, pre-filtered on the routed dept)
  -> RERANK           (cross-encoder over the shortlist)
  -> GENERATE         (LLM, grounded in the top chunks, with citations)
```

**Why transform before route?** The router is only as good as the text it reads. `"OOMKilled"`
matches no keyword and no route profile well, so a router sees noise. Rewrite it first —
*"What causes OOMKilled (a container terminated for exceeding its memory limit in
Kubernetes)…"* — and the rule stage matches `container` immediately, for free.
Exercise 14 asks you to flip the order and watch this break.

In [ ]:
def choose_transformation(query, history=None):
    """A tiny router for the transformations themselves."""
    if history and re.search(r"\b(it|this|that|they|them)\b", query, re.I):
        return "standalone"
    if len(query.split()) <= 2:
        return "rewrite"
    if re.search(r"\bboth\b|\band\b.*\b(vs|versus|compare)\b|\bboth .* and \b",
                 query, re.I):
        return "multiquery"
    if len(query.split()) > 12:
        return "stepback"
    return "none"

def transform(query, kind, history=None, entities=None):
    if kind == "rewrite":     return [rewrite_query(query)]
    if kind == "standalone":  return [make_standalone(query, history or CONVERSATION)]
    if kind == "multiquery":  return multi_query(query, entities=entities)
    if kind == "stepback":    return [query, step_back(query)]
    return [query]

In [ ]:
def ask_corp(query, history=None, entities=None, fetch_k=12, top_n=3, verbose=True):
    trace = {}

    # 1. TRANSFORM -----------------------------------------------------------
    kind = choose_transformation(query, history)
    queries = transform(query, kind, history, entities)
    trace["transformation"] = kind
    trace["queries"] = queries

    # 2. ROUTE (on the SHAPED query -- see the note above) --------------------
    dept = hybrid_router(queries[0], verbose=False)
    trace["route"] = dept

    # 3. RETRIEVE (pre-filtered on the routed department) ---------------------
    where = {"dept": dept} if dept in ROUTES else None
    pool = {}
    for q in queries:
        for h in chroma_search(q, k=fetch_k, where=where):
            pool[h["id"]] = h
    candidates = list(pool.values())
    trace["retrieved"] = len(candidates)

    # 4. RERANK --------------------------------------------------------------
    top = rerank(query, candidates, top_n=top_n) if candidates else []
    trace["top_chunks"] = [(h["id"], round(h["score"], 2)) for h in top]

    # 5. GENERATE ------------------------------------------------------------
    context = "\n\n".join(f"[{c['source']}] {c['text']}" for c in top)
    prompt = f"""Answer the question using ONLY the context below. Cite the sources.
If the context does not contain the answer, say so.

Context:
{context}

Question: {query}

Answer:"""
    answer = call_llm(prompt, task="generate", query=query, chunks=top)

    if verbose:
        show(f"ASKCORP: {query}", [
            f"transformation : {trace['transformation']}",
            f"route          : {trace['route']}",
            f"queries used   : {trace['queries']}",
            f"candidates     : {trace['retrieved']} (pre-filtered on dept={dept})",
            f"after rerank   : {trace['top_chunks']}",
            "",
            "ANSWER",
            "-" * 70,
            textwrap.fill(answer, 74),
        ])
    return answer, trace

In [ ]:
ask_corp("How many days of leave do I have left?")
ask_corp("OOMKilled")
ask_corp("Does it apply to contractors?", history=CONVERSATION)
ask_corp("Did the latency spike happen in both the Mumbai and US-East deployments?",
         entities=["the Mumbai deployment", "the US-East deployment"])
ask_corp("Why can't I pull switch association data for the rogue units on Floor 3?")

ASKCORP: How many days of leave do I have left?
transformation : none
route          : HR
queries used   : ['How many days of leave do I have left?']
candidates     : 12 (pre-filtered on dept=HR)
after rerank   : [('hr-01', 4.31), ('hr-03', 2.84), ('hr-02', -2.08)]

ANSWER
----------------------------------------------------------------------
Full-time employees accrue 24 days of paid annual leave per calendar year.
Unused leave up to 10 days may be carried into the next year.  (Sources:
hr_policy.pdf)

ASKCORP: OOMKilled
transformation : rewrite
route          : ENG
queries used   : ['What causes OOMKilled (a container being terminated for exceeding its memory limit in Kubernetes), and what are the remediation steps?']
candidates     : 12 (pre-filtered on dept=ENG)
after rerank   : [('en-03', 5.3), ('en-04', 4.14), ('en-08', -11.38)]

ANSWER
----------------------------------------------------------------------
A container exits with OOMKilled when it exceeds its memory limit.
Kuberne

('Open a pull request against the develop branch. Every PR needs one approving review, a green CI run and a linked ticket before it can be merged.\n\n(Sources: contributing.md, db_schema.md, k8s_runbook.md)',
 {'transformation': 'stepback',
  'queries': ["Why can't I pull switch association data for the rogue units on Floor 3?",
   'What is the overall architecture and set of rules governing the network monitoring system, and what data does it collect?'],
  'route': 'ENG',
  'retrieved': 12,
  'top_chunks': [('en-05', -11.12), ('en-09', -11.36), ('en-03', -11.39)]})

### 12.1 Ablation — what does each stage actually buy?

Turn the stages off one at a time and watch the top-3 change.

In [ ]:
def ablation(query, where_dept=None):
    naive   = chroma_search(query, k=3)
    routed  = chroma_search(query, k=3, where={"dept": where_dept}) if where_dept else naive
    shaped  = chroma_search(rewrite_query(query), k=3,
                            where={"dept": where_dept} if where_dept else None)
    _, tr   = ask_corp(query, verbose=False)

    show(f"Ablation: '{query}'", [
        f"{'pipeline':<38}{'top-3 chunk ids'}",
        "-" * 70,
        f"{'retrieval only':<38}{[h['id'] for h in naive]}",
        f"{'+ routing (metadata pre-filter)':<38}{[h['id'] for h in routed]}",
        f"{'+ query rewriting':<38}{[h['id'] for h in shaped]}",
        f"{'+ cross-encoder reranking (full)':<38}{[i for i, _ in tr['top_chunks']]}",
    ])

ablation("OOMKilled", where_dept="ENG")

Ablation: 'OOMKilled'
pipeline                              top-3 chunk ids
----------------------------------------------------------------------
retrieval only                        ['en-03', 'en-04', 'op-09']
+ routing (metadata pre-filter)       ['en-03', 'en-04', 'en-07']
+ query rewriting                     ['en-03', 'en-04', 'en-02']
+ cross-encoder reranking (full)      ['en-03', 'en-04', 'en-08']



---
## 13. Exercises

Work through these in order — each one reuses a function you already have.

**A. Index tuning**
1. Rerun the `nprobe` sweep with `nlist = 32` and `nlist = 512`. At what `nprobe` does each
   reach recall ≥ 0.95? What does that say about the `nlist ≈ √N` rule?
2. Rebuild `IndexPQ` with `m = 4`, `8`, `16`, `32`. Plot recall against index size. Where is
   the knee?
3. Use `pq_memory_report` to answer: 50 M vectors at `D = 1536`. How much RAM raw? With
   `m = 32`? Which FAISS index would you pick and why?

**B. Retrieval quality**
4. Add three passages of your own to `CORPUS` (any department), re-embed, re-index, and
   confirm they are retrievable.
5. Write five queries where the bi-encoder top-1 is wrong but the cross-encoder fixes it.
   If you cannot find any, what does that tell you about corpus size?
6. Implement `hybrid_retrieve(query)` that combines BM25 (`rank_bm25`) with dense retrieval
   using reciprocal rank fusion. Compare against dense-only on your five queries.

**C. Filtering**
7. Extend `chroma_search` with a `year` range filter using Chroma's `$gte` / `$lte`
   operators, then ask *"What is the insurance cover?"* restricted to `year >= 2024`.
8. Measure the pre-filter vs post-filter gap properly: for 20 queries, how often does
   post-filtering with `fetch = k` return fewer than `k` results?

**D. Transformation & routing**
9. `choose_transformation` is itself a rule-based router. Replace it with an
   embedding-based one using route profiles for the four transformation types.
10. Break `hybrid_router`: find a query where stage 1 confidently routes to the wrong
    department. Then fix it — should rules run first, or last?
11. Add a fourth department (`LEGAL`) with five passages and a route profile. Which routers
    need retraining, and which just need new sample questions?

**E. Pipeline order**
14. Flip `ask_corp` to route on the **raw** query instead of the shaped one. Which of the six
    `QUESTIONS` now routes to the wrong department, and why? Is transform-then-route always
    the right order — what does it cost you?

**F. Production**
12. Add an in-memory cache to `ask_corp` keyed on the *transformed* query. Measure the hit
    rate over the `QUESTIONS` list run twice.
13. Add a permission check: pass `user_role` into `ask_corp` and merge it into the `where`
    clause so a junior engineer cannot retrieve HR compensation passages.

---
## Summary

| You built | Function |
|---|---|
| Embeddings | `embed`, `cosine_scores`, `brute_force_search` |
| Exact search | `build_flat_ip`, `faiss_search` |
| IVF | `build_ivf`, `benchmark`, `recall_at_k` |
| PQ / IVF+PQ | `pq_memory_report`, `build_pq`, `build_ivfpq` |
| HNSW | `build_hnsw` |
| Vector database | `index_corpus`, `chroma_search`, `post_filter_search` |
| Reranking | `rerank`, `retrieve_then_rerank`, `timing_comparison` |
| Transformations | `rewrite_query`, `make_standalone`, `multi_query`, `step_back` |
| Routing | `rule_router`, `rule_router_strict`, `embedding_router`, `classifier_router`, `llm_router`, `hybrid_router` |
| Pipeline | `ask_corp`, `ablation` |

**The three ideas worth carrying out of this lab:**

1. **Every index is a trade** between recall, latency and memory. You pick the index by name
   and turn one dial (`nprobe`, `efSearch`) to move along that trade-off.
2. **Retrieval is two-stage** because precision is expensive. Recall first, cheaply;
   precision second, on a shortlist.
3. **The query the user typed is rarely the query you should search with.** Route it,
   reshape it, then retrieve.